# Import libraries

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
path = r'C:\Users\bhava\Documents\18.02.26_Instacart Basket Analysis'

In [4]:
path

'C:\\Users\\bhava\\Documents\\18.02.26_Instacart Basket Analysis'

# Importing dataset folders

In [5]:
df_ords = pd.read_csv(os.path.join(path, '02 Data', 'Prepared Data', 'orders_wrangled.csv'), index_col = False)

In [6]:
df_prods = pd.read_csv(os.path.join(path, '02 Data', 'Original Data', 'products.csv'), index_col = False)

In [7]:
df_ords.describe()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


In [8]:
df_prods.describe()

,product_id,aisle_id,department_id,prices
count,49693.000000,49693.000000,49693.000000,49693.000000
mean,24844.345139,67.770249,11.728433,9.994136
std,14343.717401,38.316774,5.850282,453.519686
min,1.000000,1.000000,1.000000,1.000000
25%,12423.000000,35.000000,7.000000,4.100000
50%,24845.000000,69.000000,13.000000,7.100000
75%,37265.000000,100.000000,17.000000,11.200000
max,49688.000000,134.000000,21.000000,99999.000000


# Mixed Data

# Always check for these mixed-type columns before moving forward with any analytical work, as they can break functions and generally cause problems in your procedures

# Creating a test example to understand mixed data

In [9]:
df_test = pd.DataFrame()

In [10]:
df_test['mix']=['a', 'b', 1, True]

In [11]:
df_test.head()

,mix
0,a
1,b
2,1
3,True


# note this function for later - used to test if the df contains any mixed-type data, in a for-loop way for each column

In [12]:
for col in df_test.columns.tolist():
  weird = (df_test[[col]].applymap(type) != df_test[[col]].iloc[0].apply(type)).any(axis = 1)
  if len (df_test[weird]) > 0:
    print (col)

mix


C:\Users\bhava\AppData\Local\Temp\ipykernel_15312\1786467950.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  weird = (df_test[[col]].applymap(type) != df_test[[col]].iloc[0].apply(type)).any(axis = 1)


In [13]:
df_test['mix'] = df_test['mix'].astype('str')

# Missing Values

In [14]:
df_prods.head()

,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.8
1,2,All-Seasons Salt,104,13,9.3
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.5
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.5
4,5,Green Chile Anytime Sauce,5,13,4.3


In [15]:
df_prods.isnull().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

In [16]:
df_prods.isnull()

#not very helpful 

,product_id,product_name,aisle_id,department_id,prices
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
...,...,...,...,...,...
49688,False,False,False,False,False
49689,False,False,False,False,False
49690,False,False,False,False,False
49691,False,False,False,False,False


In [17]:
# Create a subset df_nan

df_nan = df_prods[df_prods['product_name'].isnull()==True]

In [18]:
df_nan

,product_id,product_name,aisle_id,department_id,prices
33,34,NaN,121,14,12.2
68,69,NaN,26,7,11.8
115,116,NaN,93,3,10.8
261,262,NaN,110,13,12.1
525,525,NaN,109,11,1.2
1511,1511,NaN,84,16,14.3
1780,1780,NaN,126,11,12.3
2240,2240,NaN,52,1,14.2
2586,2586,NaN,104,13,12.4
3159,3159,NaN,126,11,13.1


In [19]:
df_nan.describe()

,product_id,aisle_id,department_id,prices
count,16.000000,16.000000,16.000000,16.000000
mean,6684.000000,89.937500,10.937500,13.012500
std,12836.665242,33.731229,4.639953,3.881731
min,34.000000,26.000000,1.000000,1.200000
25%,459.250000,70.750000,7.750000,12.175000
50%,2413.000000,98.500000,11.500000,13.650000
75%,3872.750000,120.000000,14.500000,14.425000
max,40440.000000,126.000000,16.000000,20.900000


## Imputing Data: Replacing missing values with Mean or Median

df['column with missings'].fillna(mean value, inplace=True)

df['column with missings'].fillna(median value, inplace=True)

## Another method for imputing missing values = Linear Interpolation 

##  String values can’t be imputed like numeric values. You can either remove the missing values entirely or filter out the ones that aren’t missing into a subset dataframe and continue your analysis with this new dataframe.

In [20]:
df_prods.shape

(49693, 5)

In [21]:
# df_prods_clean 

df_prods_clean = df_prods[df_prods['product_name'].isnull() == False]

In [38]:
df_prods_clean.shape

(49677, 5)

# Another method to remove missing values 

df_prods.dropna(inplace = True)

# And from columns 

df_prods.dropna(subset = [‘product_name’], inplace = True)


If you don’t specify an inplace argument in your code, the function will take the default setting, which is inplace = False. When specified as False, the command will only return a view of the changed dataframe, leaving the original dataframe untouched.

# Duplicates

In [22]:
df_dups = df_prods_clean[df_prods_clean.duplicated()]

In [23]:
df_dups

,product_id,product_name,aisle_id,department_id,prices
462,462,Fiber 4g Gummy Dietary Supplement,70,11,4.8
18459,18458,Ranger IPA,27,5,9.2
26810,26808,Black House Coffee Roasty Stout Beer,27,5,13.4
35309,35306,Gluten Free Organic Peanut Butter & Chocolate ...,121,14,6.8
35495,35491,Adore Forever Body Wash,127,11,9.9


#Addressing duplicates

df.drop_duplicates()

In [24]:
df_prods_clean.shape

(49677, 5)

In [25]:
df_prods_clean_no_dups = df_prods_clean.drop_duplicates()

In [26]:
df_prods_clean_no_dups.shape

(49672, 5)

# Export

In [27]:
df_prods_clean_no_dups.to_csv(os.path.join(path, '02 Data','Prepared Data', 'products_checked.csv'))

# Task 

## 2. Interpreting Data

In [28]:
df_ords.head(10)

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0
2,473747,1,3,3,12,21.0
3,2254736,1,4,4,7,29.0
4,431534,1,5,4,15,28.0
5,3367565,1,6,2,7,19.0
6,550135,1,7,1,9,20.0
7,3108588,1,8,1,14,14.0
8,2295261,1,9,1,16,0.0
9,2550362,1,10,4,8,30.0


In [29]:
df_ords.tail(10)

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
3421073,2307371,206209,5,4,15,3.0
3421074,3186442,206209,6,0,16,3.0
3421075,550836,206209,7,2,13,9.0
3421076,2129269,206209,8,3,17,22.0
3421077,2558525,206209,9,4,15,22.0
3421078,2266710,206209,10,5,18,29.0
3421079,1854736,206209,11,4,10,30.0
3421080,626363,206209,12,1,12,18.0
3421081,2977660,206209,13,1,12,7.0
3421082,272231,206209,14,6,14,30.0


In [30]:
df_ords.describe()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


## Overall, the data appears largely consistent, but the presence of zero-day reorder intervals is slightly unexpected and could be explored further to ensure there are no underlying data quality issues.

# 3. and 4. Mixed Data

In [31]:
df_ords.head()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0
2,473747,1,3,3,12,21.0
3,2254736,1,4,4,7,29.0
4,431534,1,5,4,15,28.0


In [32]:
df_ords.tail()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
3421078,2266710,206209,10,5,18,29.0
3421079,1854736,206209,11,4,10,30.0
3421080,626363,206209,12,1,12,18.0
3421081,2977660,206209,13,1,12,7.0
3421082,272231,206209,14,6,14,30.0


In [33]:
df_ords.shape

(3421083, 6)

In [34]:
df_ords.dtypes

order_id                   int64
user_id                    int64
order_number               int64
order_day_of_week          int64
order_hour_of_day          int64
days_since_last_order    float64
dtype: object

In [60]:
df_ords.head()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
0,2539329,1,1,2,8,11.11484
1,2398795,1,2,3,7,15.00000
2,473747,1,3,3,12,21.00000
3,2254736,1,4,4,7,29.00000
4,431534,1,5,4,15,28.00000


# No mixes observed

# 5. Missing Values 

In [35]:
df_ords.isnull().sum()

order_id                      0
user_id                       0
order_number                  0
order_day_of_week             0
order_hour_of_day             0
days_since_last_order    206209
dtype: int64

## Answer: 206209 observations of rows contain no values in the column days_since_last_order

This could be either because no previous order exists, due to incomplete data entry or import issues or data type or formatting issues. Sometimes certain rows are exempt from tracking, e.g., test orders, guest users, or imported legacy data.

In [36]:
df_nan_ords = df_ords[df_ords['days_since_last_order'].isnull() == True]

In [37]:
df_ords.describe()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


In [38]:
df_nan_ords.describe()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
count,2.062090e+05,206209.000000,206209.0,206209.000000,206209.000000,0.0
mean,1.708462e+06,103105.000000,1.0,2.754118,13.626597,NaN
std,9.881299e+05,59527.555167,0.0,2.076205,4.223769,NaN
min,2.000000e+01,1.000000,1.0,0.000000,0.000000,NaN
25%,8.507300e+05,51553.000000,1.0,1.000000,11.000000,NaN
50%,1.706246e+06,103105.000000,1.0,3.000000,14.000000,NaN
75%,2.564292e+06,154657.000000,1.0,5.000000,17.000000,NaN
max,3.421081e+06,206209.000000,1.0,6.000000,23.000000,NaN


In [39]:
df_nan_ords

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
0,2539329,1,1,2,8,NaN
11,2168274,2,1,2,11,NaN
26,1374495,3,1,1,14,NaN
39,3343014,4,1,6,11,NaN
45,2717275,5,1,3,12,NaN
...,...,...,...,...,...,...
3420930,969311,206205,1,4,12,NaN
3420934,3189322,206206,1,3,18,NaN
3421002,2166133,206207,1,6,19,NaN
3421019,2227043,206208,1,1,15,NaN


# Imputing with Average

In [40]:
df_ords['days_since_last_order'] = df_ords['days_since_last_order'].fillna(1.111484e+01)

In [41]:
df_ords.head()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
0,2539329,1,1,2,8,11.11484
1,2398795,1,2,3,7,15.00000
2,473747,1,3,3,12,21.00000
3,2254736,1,4,4,7,29.00000
4,431534,1,5,4,15,28.00000


In [54]:
df_ords.tail()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
3421078,2266710,206209,10,5,18,29.0
3421079,1854736,206209,11,4,10,30.0
3421080,626363,206209,12,1,12,18.0
3421081,2977660,206209,13,1,12,7.0
3421082,272231,206209,14,6,14,30.0


In [43]:
df_ords_clean = df_ords['days_since_last_order'] = df_ords['days_since_last_order'].fillna(1.111484e+01)

In [44]:
df_ords_clean.head()

0    11.11484
1    15.00000
2    21.00000
3    29.00000
4    28.00000
Name: days_since_last_order, dtype: float64

In [45]:
df_ords_clean.describe()

count    3.421083e+06
mean     1.111484e+01
std      8.924952e+00
min      0.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.500000e+01
max      3.000000e+01
Name: days_since_last_order, dtype: float64

In [46]:
df_ords_clean.shape

(3421083,)

## 6. Answer/Reasoning: Here, I chose the method to impute the missing data in the column day_since_last_order, with the mean of the dataset column because the column contained a considerable number of missing values only in this column. Also, removing such a large number of rows would mess with the descriptive analysis of the remaining columns as well. After imputation with mean, the max and min in the column day_since_last_order has remained the same, with only minor changes to the other descriptive sections. 

# 7. Duplicate Values 

In [47]:
df_ords_clean_no_dups = df_ords[df_ords.duplicated()]

In [52]:
df_ords_clean_no_dups.describe()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order
count,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
df_ords_clean_no_dups.shape

(0, 6)

## 8. Answer: No duplicates found in the dataset

# 9. Exporting

In [55]:
df_ords_clean_no_dups.to_csv(os.path.join(path, '02 Data','Prepared Data', 'orders_checked.csv'))

In [57]:
df_prods_clean_no_dups.to_csv(os.path.join(path, '02 Data','Prepared Data', 'products_checked.csv'))